# Eora parser walkthrough

This notebook is the practical guide for parsing the Eora datasets supported by MARIO.


<div class="admonition warning">
<div class="admonition-title">Warning</div>
At the moment, MARIO supports only <strong>EORA1</strong>.
<br><br>
Support for <strong>EORA2</strong> is planned as soon as possible.
<br><br>
EORA website: <a href="https://www.eora.org">https://www.eora.org</a>
</div>

## What this notebook covers

- which Eora workflows MARIO supports and which it does not;
- the difference between `Eora26` and single-region local files;
- which files are required for the `Eora26` workflow;
- how `multi_region=`, `indeces=`, `country=`, and `price=` are used;
- how `name_convention=` and `aggregate_trade=` affect the single-region workflow;
- which parser caveats matter in practice.


## Important scope note

MARIO does **not** parse the full Eora MRIO release.

The supported workflows are only:

- `Eora26` as the multi-region workflow;
- local single-region Eora tables.


## Relevant source page

- Official Eora website: [Eora MRIO portal](https://www.worldmrio.com/)

Automatic download is not part of the current MARIO workflow. The expected path is to work from local files that you already downloaded.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_eora(...)`

The same function supports:

- `Eora26` with `multi_region=True`;
- local single-region Eora files with `multi_region=False`.


## Key arguments

The key public arguments are:

- `path`: one Eora file or one local dataset directory;
- `multi_region`: use `True` for `Eora26` and `False` for local single-region files;
- `table`: mainly relevant for single-region parsing;
- `indeces`: optional path to the `Eora26` label files;
- `name_convention`: `full_name` or `abbreviation` in the single-region workflow;
- `aggregate_trade`: whether to collapse detailed import/export rows in the single-region workflow;
- `country`: country selector when a directory contains multiple single-region files;
- `price`: optional selector when a single-region directory contains multiple price variants.


## `Eora26` versus single-region Eora

Use `multi_region=True` only for `Eora26`. This workflow expects the standard `Eora26_<year>_<price>_*.txt` files plus the `labels_*.txt` files.

Use `multi_region=False` for the local single-region tables. In that case MARIO resolves one file such as `IO_ITA_2015_BasicPrice.txt` and can usually infer whether it behaves like an `IOT` or a `SUT`.


## Label files for `Eora26`

For `Eora26`, MARIO needs both:

- the numeric files such as `T`, `FD`, `VA`, `Q`, and `QY`;
- the label files such as `labels_T.txt`, `labels_FD.txt`, `labels_VA.txt`, and `labels_Q.txt`.

If the label files already live next to the dataset files, `indeces=` can be omitted. Otherwise, point `indeces=` to the directory that contains them.


## Local layout examples

There is no dedicated Eora 2 parser yet; current MARIO support is still limited to the Eora 1 style workflows covered here.

`Eora26` expects a directory like:

```text
EORA/
├── Eora26_2017_bp/
│   ├── Eora26_2017_bp_T.txt
│   ├── Eora26_2017_bp_FD.txt
│   ├── Eora26_2017_bp_VA.txt
│   ├── Eora26_2017_bp_Q.txt
│   └── Eora26_2017_bp_QY.txt
└── indices/
    ├── labels_T.txt
    ├── labels_FD.txt
    ├── labels_VA.txt
    └── labels_Q.txt
```

For single-region workflows, `path` can point to one file or to one directory of files named like `IO_ITA_2015_BasicPrice.txt`. Use `country=` and `price=` when the directory contains multiple candidates.

In [1]:
import mario

## Parse one `Eora26` directory

This is the supported multi-region EORA26 workflow.


In [ ]:
db = mario.parse_eora(
    path="/path/to/Eora26_2017_bp",
    multi_region=True,
)

db

INFO Parser: reading Eora26 from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/a-Datasets/_Input Output Databases/EORA/EORA1/Eora26/Eora26_2015_bp.
INFO Parser: Eora26 parsed with 189 regions, 26 sectors, 7 factor rows and 2728 extension rows.
INFO Metadata: initialized.


name = Eora26_2015_bp
table = IOT
scenarios = ['baseline']
Factor of production = 7
Satellite account = 2728
Consumption category = 7
Region = 189
Sector = 26

## Parse one local single-region file

This is the non-multi-region Eora workflow. When `path` points to a directory, `country=` selects the file to parse.


In [ ]:
db = mario.parse_eora(
    path="/path/to/IO_ITA_2017_BasicPrice.txt",
    multi_region=False,
)

db

INFO Parser: reading EORA single-region file IO_ITA_2017_BasicPrice.txt.
INFO Parser: EORA single-region parsed as SUT with 7 factor rows and 2631 extension rows.
INFO Metadata: initialized.


name = IO_ITA_2017_BasicPrice
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 61
Commodity = 61
Factor of production = 7
Satellite account = 2631
Consumption category = 7
Region = 1

## Select one price variant in the single-region workflow

Use `price=` when the directory contains more than one variant for the same country and year.


In [ ]:
db = mario.parse_eora(
    path="/path/to/IO_All_2017",
    multi_region=False,
    country="ITA",
    price="BasicPrice",
)

db

INFO Parser: reading EORA single-region file IO_ITA_2017_BasicPrice.txt.
INFO Parser: EORA single-region parsed as SUT with 7 factor rows and 2631 extension rows.
INFO Metadata: initialized.


name = IO_ITA_2017_BasicPrice
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 61
Commodity = 61
Factor of production = 7
Satellite account = 2631
Consumption category = 7
Region = 1

## Aggregate trade rows in the single-region workflow

Use `aggregate_trade=True` when you want imports and exports collapsed into totals instead of preserving bilateral trade labels.


In [ ]:
db = mario.parse_eora(
    path="/path/to/IO_All_2017",
    multi_region=False,
    country="ITA",
    price="BasicPrice",
    aggregate_trade=True,
)

db

INFO Parser: reading EORA single-region file IO_ITA_2017_BasicPrice.txt.
INFO Parser: EORA single-region parsed as SUT with 7 factor rows and 2631 extension rows.
INFO Metadata: initialized.


name = IO_ITA_2017_BasicPrice
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 61
Commodity = 61
Factor of production = 7
Satellite account = 2631
Consumption category = 7
Region = 1

## Notes and caveats

- there is no parser here for the full Eora MRIO release;
- multi-region means `Eora26` only;
- the `Eora26` parser applies a few consistency repairs during parsing; those notes are stored in metadata;
- single-region parsing can infer `IOT` versus `SUT` automatically from the local file structure.


In [10]:
db.meta_history

[2026-05-31 08:46:49]    Table added into metadata with value equal to SUT.
[2026-05-31 08:46:49]    Price added into metadata with value equal to BasicPrice.
[2026-05-31 08:46:49]    Source added into metadata with value equal to Eora website @ https://www.worldmrio.com/.
[2026-05-31 08:46:49]    Year added into metadata with value equal to 2017.
[2026-05-31 08:46:49]    Tech_Assumption added into metadata with value equal to industry-based.


In [11]:
db.sets

['Activity',
 'Commodity',
 'Factor of production',
 'Satellite account',
 'Consumption category',
 'Region']

In [12]:
db.units

{'Activity':                                                                unit
Agriculture, hunting and related service activi...  current 000 US$
Forestry, logging and related service activities    current 000 US$
Fishing, operating of fish hatcheries and fish ...  current 000 US$
Mining of coal and lignite; extraction of peat      current 000 US$
Extraction of crude petroleum and natural gas; ...  current 000 US$
...                                                             ...
Recreational, cultural and sporting activities      current 000 US$
Other service activities                            current 000 US$
Private households with employed persons            current 000 US$
FISIM                                               current 000 US$
Re-export                                           current 000 US$

[61 rows x 1 columns], 'Commodity':                                                                unit
Products of agriculture, hunting and related se...  current 000 US